In this notebook we test Vanguri et al. architecture on the I3LUNG data. 


# Set-up

In [1]:
import pandas as pd
import os
from lung_helpers import train, get_training_data, train_standard_split, hyperparameter_search

In [2]:
if not 'summary_dfs' in locals():
    print ("init summary_dfs")
    summary_dfs = {}

init summary_dfs


In [3]:
DATA_DIR = "../data"
RESULTS_DIR = "./results"

os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
df_outcomes = pd.read_csv(f"{DATA_DIR}/outcomes.csv")
df_outcomes = df_outcomes.rename(columns={'Subject': 'main_index'}).set_index('main_index')

# Find coumns with 2075 label non-null
outcome_cols_complete = []
for col in df_outcomes.columns:
    n_valid = df_outcomes[col].notna().sum()
    print(f"{col}: {n_valid} non-null")
    if n_valid == 2075:
        outcome_cols_complete.append(col)

print(f"\noutcomes with all 2075 labels: {outcome_cols_complete}")

DEATH EVENT: 2075 non-null
PROGRESSION EVENT: 2070 non-null
THERAPY END EVENT: 2075 non-null
BEST RESPONSE: 2068 non-null
IO LINE: 2075 non-null
IO IOCHT: 2075 non-null
HISTOLOGY SQUAMOUS: 2052 non-null
HISTOLOGY ADENOCARCINOMA: 2052 non-null
ORR: 2068 non-null
DCR: 2068 non-null
CBR: 2066 non-null
PFS MONTHS: 2066 non-null
OS MONTHS: 2074 non-null
TTF MONTHS: 2073 non-null
OS_6: 1972 non-null
OS_24: 1699 non-null

outcomes with all 2075 labels: ['DEATH EVENT', 'THERAPY END EVENT', 'IO LINE', 'IO IOCHT']


In [5]:
# complete cohort
df_clinical_full = pd.read_csv(f"{DATA_DIR}/rwd_processed.csv")
df_clinical_full = df_clinical_full.rename(columns={'Subject': 'main_index'})

# Cohort2 (IO LINE == 1)
df_rwd = pd.read_csv(f"{DATA_DIR}/rwd.csv")
df_rwd_cohort2 = df_rwd[df_rwd['IO LINE'] == 1].copy()

cohorts = {
    'cohort23': {
        'subjects': df_clinical_full['main_index'].values,
        'split_source': df_clinical_full
    },
    'cohort2': {
        'subjects': df_rwd_cohort2['Subject'].values,
        'split_source': df_rwd_cohort2.rename(columns={'Subject': 'main_index'})
    }
}

all_splits = {}

for cohort_name, cohort_info in cohorts.items():
    print(f"\n{'='*50}")
    print(f"Processing {cohort_name}")
    print(f"{'='*50}")
    
    # filter df_clinical_full for the subjects in the cohort
    cohort_subjects = cohort_info['subjects']
    df_clinical = df_clinical_full[df_clinical_full['main_index'].isin(cohort_subjects)].copy()
    
    # extract splits
    split_df = cohort_info['split_source']
    train_df = split_df[split_df['SET'] == 'TRAIN']
    test_df = split_df[split_df['SET'] == 'TEST']
    ext_val_df = split_df[split_df['SET'] == 'EXVAL']
    
    train_px = train_df['main_index'].values
    test_px = test_df['main_index'].values
    ext_val_px = ext_val_df['main_index'].values
    
    print(f"=== INITIAL SPLIT {cohort_name} ===")
    print(f"Train: {len(train_px)}, Test: {len(test_px)}, ExtVal: {len(ext_val_px)}")
    
    # extract predefined folds (exclude UOC from training centers)
    predefined_folds = train_df[train_df['CENTER'] != 'UOC'][['main_index', 'CENTER']].rename(columns={'CENTER': 'fold'})
    predefined_folds = predefined_folds.set_index('main_index')
    
    print(f"Train Centers (per CV): {predefined_folds['fold'].unique()}")
    print(f"Patients for center:\n{predefined_folds['fold'].value_counts()}")
    
    # prepare clinical dataframe
    df_clinical = df_clinical.set_index('main_index')
    df_clinical = df_clinical.drop(columns=['CENTER', 'SET'], errors='ignore')
    
    # save for later use
    all_splits[cohort_name] = {
        'df_clinical': df_clinical,
        'train_px': train_px,
        'test_px': test_px,
        'ext_val_px': ext_val_px,
        'predefined_folds': predefined_folds
    }


Processing cohort23
=== INITIAL SPLIT cohort23 ===
Train: 1550, Test: 274, ExtVal: 251
Train Centers (per CV): ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
Patients for center:
fold
INT     582
GHD     365
SZMC    232
MH      196
VHIO    175
Name: count, dtype: int64

Processing cohort2
=== INITIAL SPLIT cohort2 ===
Train: 1046, Test: 181, ExtVal: 210
Train Centers (per CV): ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
Patients for center:
fold
INT     386
GHD     231
SZMC    193
MH      164
VHIO     72
Name: count, dtype: int64


In [ ]:
df_pyrad = pd.read_csv(f"{DATA_DIR}/pyradiomics.csv")
df_pyrad = df_pyrad.rename(columns={'Subject': 'main_index'})

print(f"total patients: {len(df_pyrad)}")

# Set index e drop colonne
df_pyrad = df_pyrad.set_index('main_index')
df_pyrad = df_pyrad.drop(columns=['CENTER', 'SET'], errors='ignore')

print(f"pyradiomics features: {len(df_pyrad.columns)}")

total patients: 877
pyradiomics features: 128


In [ ]:
df_fmrad = pd.read_csv(f"{DATA_DIR}/fmrad.csv")
df_fmrad = df_fmrad.rename(columns={'Subject': 'main_index'})

print(f"total patients: {len(df_fmrad)}")
# Set index e drop columns
df_fmrad = df_fmrad.set_index('main_index')
df_fmrad = df_fmrad.drop(columns=['CENTER', 'SET'], errors='ignore')

print(f"fmradiomics features: {len(df_fmrad.columns)}")

total patients: 896
fmradiomics features: 4096


In [ ]:
df_pathology = pd.read_csv("f{DATA_DIR}/digital_pathology_processed.csv")  
df_pathology = df_pathology.rename(columns={'Subject': 'main_index'})  

print(f"total patients: {len(df_pathology)}")

# Set index e drop columns
df_pathology = df_pathology.set_index('main_index')
df_pathology = df_pathology.drop(columns=['CENTER', 'SET'], errors='ignore')  

print(f"pathology features: {len(df_pathology.columns)}")

total patients: 846
pathology features: 768


In [ ]:
df_genomics = pd.read_csv(f"{DATA_DIR}/genomics_processed.csv") 
df_genomics = df_genomics.rename(columns={'Subject': 'main_index'})  

print(f"total patients: {len(df_genomics)}")

# Set index e drop columns
df_genomics = df_genomics.set_index('main_index')
df_genomics = df_genomics.drop(columns=['CENTER', 'SET'], errors='ignore')  #

print(f"genomics features: {len(df_genomics.columns)}")

total patients: 1705
genomics features: 4


In [ ]:
df_outcomes = pd.read_csv(f"{DATA_DIR}/outcomes.csv")
df_outcomes = df_outcomes.rename(columns={'Subject': 'main_index'})
df_outcomes = df_outcomes.set_index('main_index')
df_outcomes = df_outcomes[~df_outcomes.index.duplicated()]
print("total patients:", df_outcomes.index.nunique())
print(f"total outcomes: {len(df_outcomes.columns)}")

total patients: 2075
total outcomes: 16


# Hyperparam search

default hyperparam from original paper

In [ ]:
# for multimodal models, attention gate
model_params = {'epochs':125, 'lr':0.01, 'alpha':0.001, 'beta':0.0, 'cross_modality_enabled':False}
# for unimodal models, no attention gate
model_params_gate_off = {'epochs':125, 'lr':0.01, 'alpha':0.001, 'beta':0.0, 'cross_modality_enabled':False, 'attention_gate_enabled':False}

hyperparam grid

In [13]:
hyperparam_grid = {
    'epochs': [100, 125, 150],
    'lr': [0.001, 0.01, 0.1],
    'alpha': [0.0001, 0.001, 0.01],  # L2 weight
    'beta': [0.0, 0.001, 0.01]       # AR2 weight
}

In [ ]:
# Prepare data for cohort23, OS_6
cohort_name = 'cohort23'
outcome_col = 'OS_6'

df_clinical_clean = all_splits[cohort_name]['df_clinical'].copy()
train_px = all_splits[cohort_name]['train_px']

# Filtra solo train set
df_clinical_train = df_clinical_clean.loc[train_px]

cohort_subjects = df_clinical_train.index
df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()

df_dyam = df_clinical_train.copy()
df_dyam['label'] = df_outcomes[outcome_col]
df_dyam = df_dyam.dropna(subset=['label'])

X_clin = df_dyam.drop(columns=['label']).fillna(0)
X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)

modality_mask = pd.DataFrame({
    'clinical': 1,
    'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
    'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
    'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
}, index=df_dyam.index)

df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)

# L1 filter
df_rad_for_filter = X_rad.reset_index()
df_rad_for_filter['job_tag'] = 'filtered-radiomics'
df_rad_for_filter['site'] = 'PC'
df_rad_for_filter['lesion_index'] = 1

dfs_filters = {
    1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
}

# Run hyperparameter search
best_params, search_results = hyperparameter_search(
    [X_clin, X_rad, X_path, X_gen],
    modality_mask, 
    df_out,
    dfs_filters,
    hyperparam_grid
)

# Save results
search_results.to_csv('/home/ludovica/vanguri/results/hyperparam_search_results_cohort23_OS_6.csv', index=False)

print(f"\nUsing best parameters for all models: {best_params}")

Testing 81 combinations with 10-fold CV


Starting cross-validation training
predefined_folds: False
  0%|          | 0/10 [00:00<?, ?it/s]


Processing fold 1: 1325 train samples, 148 validation samples
 Applying L1 filter on modality position 1
Selected 51 outlier-stable features
Fit called with 4 modalities, 1325 samples
Modality 0: shape (1325, 11), noscale=False
Modality 1: shape (1325, 12), noscale=False
Modality 2: shape (1325, 768), noscale=False
Modality 3: shape (1325, 4), noscale=False
Mask shape: torch.Size([1325, 4]), available modalities per patient: 2.61
Class balance: 891.0/1325 positive
Training 100 epochs with 6 batches
Final output - min: -0.9656, max: 0.9347, mean: 0.0036
Training mode - mu: -0.0123, std: 0.3697
Eval mode - using mu: -0.0123, std: 0.3697
 10%|█         | 1/10 [00:12<01:55, 12.78s/it]
Processing fold 2: 1325 train samples, 148 validation samples
 Applying L1 filter on modality position 1
Selected 53 outlier-stable features
Fit called with 4 modalities, 1325 samples
Modality 0: shape (1325, 11), noscale=False
Modality 1: shape (1325, 12), noscale=False
Modality 2: shape (1325, 768), noscal

# Unimodal clinical

cross validation unimodal

In [15]:
import os

os.makedirs('/home/ludovica/vanguri/results', exist_ok=True)

df_outcomes = pd.read_csv("/home/ludovica/I3LUNG_PDSS/data/outcomes.csv")
df_outcomes = df_outcomes.rename(columns={'Subject': 'main_index'})
df_outcomes = df_outcomes.set_index('main_index')
df_outcomes = df_outcomes[~df_outcomes.index.duplicated()]

summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training models for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        X_dyam = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_dyam.isna().sum().sum()}")
        
        modality_mask_single = pd.DataFrame({
            'clinical': X_dyam.notna().any(axis=1).astype(int)
        }, index=X_dyam.index)
        print(f"Modality mask created with {modality_mask_single['clinical'].sum()} samples having clinical data")
        
        X_dyam_filled = X_dyam.fillna(0)
        print(f"After filling missing values: {X_dyam_filled.isna().sum().sum()} missing values")
        
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        common_index = X_dyam_filled.index & modality_mask_single.index & df_out.index
        X_dyam_filled = X_dyam_filled.loc[common_index]
        modality_mask_single = modality_mask_single.loc[common_index]
        df_out = df_out.loc[common_index]
        
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        cv_index = predefined_folds_aligned.index
        X_dyam_cv = X_dyam_filled.loc[cv_index]
        modality_mask_cv = modality_mask_single.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'], _ = train(
            modality_list_in=[X_dyam_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter={},
            model_params=model_params_gate_off,
            predefined_folds=predefined_folds_aligned
        )
        
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training models for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Modality mask created with 1699 samples having clinical data
After filling missing values: 0 missing values
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC']


Processing fold 1: 921 train samples, 362 validation samples
 20%|██        | 1/5 [00:02<00:09,  2.37s/it]
Processing fold 2: 773 train samples, 510 validation samples
 40%|████      | 2/5 [00:03<00:05,  1.80s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 60%|██████    | 3/5 [00:05<00:03,  1.94s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 80%|████████  | 4/5 [00:07<00:01,  1.86s/it]
Processing fold 5: 1119 train samples, 164 validation samples
100%|██████████| 5/5 [00:09<00:00,  1.92s/it]

Overall AUC calculated on 1283 samples (pos=324, neg=959)
AUC = 0.699 (95% CI: 0.665-0.734)

--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Modality mask created with 1972 samples having clinical data
After filling missing values: 0 missing values
Predefined folds after alignment: 1473 samples
Folds available: ['GHD' 'INT' 'MH' '

train/test/ext-val unimodal

In [16]:
import os

# All'inizio, prima del loop
os.makedirs('/home/ludovica/vanguri/results', exist_ok=True)

df_outcomes = pd.read_csv("/home/ludovica/I3LUNG_PDSS/data/outcomes.csv")
df_outcomes = df_outcomes.rename(columns={'Subject': 'main_index'})
df_outcomes = df_outcomes.set_index('main_index')
df_outcomes = df_outcomes[~df_outcomes.index.duplicated()]

summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training models for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Feature matrix
        X_dyam = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_dyam.isna().sum().sum()}")
        
        # Modality mask
        modality_mask_single = pd.DataFrame({
            'clinical': X_dyam.notna().any(axis=1).astype(int)
        }, index=X_dyam.index)
        print(f"Modality mask created with {modality_mask_single['clinical'].sum()} samples having clinical data")
        
        # Fill missing
        X_dyam_filled = X_dyam.fillna(0)
        print(f"After filling missing values: {X_dyam_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_dyam_filled.index & modality_mask_single.index & df_out.index
        X_dyam_filled = X_dyam_filled.loc[common_index]
        modality_mask_single = modality_mask_single.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # Train
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_dyam_filled],
            modality_mask=modality_mask_single,
            outcomes=df_out,
            l1_dfs_filter={},
            model_params=model_params_gate_off,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Salva predizioni
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training models for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Modality mask created with 1699 samples having clinical data
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190

Test AUC: 0.713 (95% CI: 0.634-0.792)
External Validation AUC: 0.592 (95% CI: 0.524-0.660)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Modality mask created with 1972 samples having clinical data
After filling missing values: 0 missing values
After filtering - Train: 1473, Test: 259, ExtVal: 240
Train samples: 1473
Test samples: 259
External validation samples: 240

Test AUC: 0.684 (95% CI: 0.614-0.754)
External Validat

# Bimodal Clinical + Radiomics

bimodal clinical + pyrad cross validation

In [17]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']

    print(f"Clinical samples for {cohort_name}: {len(df_clinical_clean)}")
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()

    print(f"Radiology samples for {cohort_name}: {len(df_radiology_clean)}")
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad.fillna(0)
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align / saftey check 
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        print(f"Common index samples: {len(common_index)}")
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]

        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")

        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue


        # Filter for CV 
        X_clin_cv = X_clin_filled.loc[predefined_folds_aligned.index]
        X_rad_cv = X_rad_filled.loc[predefined_folds_aligned.index]
        modality_mask_cv = modality_mask_bimodal.loc[predefined_folds_aligned.index]
        df_out_cv = df_out.loc[predefined_folds_aligned.index]

        
        # L1 filter
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_rad_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_pyrad_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models for cohort23
Clinical samples for cohort23: 2075
Radiology samples for cohort23: 877

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
Common index samples: 1699
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<?, ?it/s


Processing fold 1: 921 train samples, 362 validation samples
 Applying L1 filter on modality position 1
Selected 61 outlier-stable features
 20%|██        | 1/5 [00:01<00:07,  1.91s/it]
Processing fold 2: 773 train samples, 510 validation samples
 Applying L1 filter on modality position 1
Selected 67 outlier-stable features
 40%|████      | 2/5 [00:03<00:05,  1.75s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 60 outlier-stable features
 60%|██████    | 3/5 [00:05<00:04,  2.04s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 Applying L1 filter on modality position 1
Selected 54 outlier-stable features
 80%|████████  | 4/5 [00:08<00:02,  2.19s/it]
Processing fold 5: 1119 train samples, 164 validation samples
 Applying L1 filter on modality position 1
Selected 58 outlier-stable features
100%|██████████| 5/5 [00:10<00:00,  2.11s/it]

Overall AUC calculated on 1283 samples (pos=324, neg=959)
AUC = 0

bimodal clinical + pyrad train/test/ext_val split 

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filter setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled],
            modality_mask=modality_mask_bimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_rad_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_pyrad_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 61 outlier-stable features

Test AUC: 0.695 (95% CI: 0.610-0.779)
External Validation AUC: 0.546 (95% CI: 0.460-0.631)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 859
After filling missing values: 0 missing values
After filtering - Tra

bimodal clinical + fmrad cross validation

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (FM radiology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        modality_mask_cv = modality_mask_bimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filter
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_rad_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_radfm_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (FM radiology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 350
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<?, ?it/s]
Processing fold 1: 921 train samples, 362

bimodal clinical + fmrad train/test/ext_val

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (FM radiology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filter setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled],
            modality_mask=modality_mask_bimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_rad_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_radfm_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (FM radiology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 350
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 2704 outlier-stable features

Test AUC: 0.702 (95% CI: 0.621-0.783)
External Validation AUC: 0.556 (95% CI: 0.495-0.618)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 390
After filling missing values: 0 missing values
After

# Bimodal Clinical + Digital Pathology

bimodal clinical + dp cross validation

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (pathology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features pathology (allineato con reindex)
        X_dp = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with pathology data: {modality_mask_bimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_dp_filled = X_dp
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_dp_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_dp_filled = X_dp_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_dp_cv = X_dp_filled.loc[cv_index]
        modality_mask_cv = modality_mask_bimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_dp_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter={},
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (pathology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with pathology data: 370
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<?, ?it/s]


Processing fold 1: 921 train samples, 362 validation samples
 20%|██        | 1/5 [00:05<00:20,  5.06s/it]
Processing fold 2: 773 train samples, 510 validation samples
 40%|████      | 2/5 [00:09<00:14,  4.96s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 60%|██████    | 3/5 [00:15<00:10,  5.12s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 80%|████████  | 4/5 [00:21<00:05,  5.41s/it]
Processing fold 5: 1119 train samples, 164 validation samples
100%|██████████| 5/5 [00:25<00:00,  5.18s/it]

Overall AUC calculated on 1283 samples (pos=324, neg=959)
AUC = 0.670 (95% CI: 0.636-0.705)

--- Processing outcome: OS_6 for cohort23 ---
Patients with OS_6 label: 1972
Total patients: 1972
Patients with clinical data: 1972
Patients with pathology data: 434
Predefined folds after alignment: 1473 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1473 samples for CV

Starting cross-validation training
predefined_folds: True
Usin

bimodal clinical + dp standard

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (pathology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features pathology (allineato con reindex)
        X_dp = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with pathology data: {modality_mask_bimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_dp_filled = X_dp
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_dp_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_dp_filled = X_dp_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_dp_filled],
            modality_mask=modality_mask_bimodal,
            outcomes=df_out,
            l1_dfs_filter={},
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (pathology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with pathology data: 370
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190

Test AUC: 0.696 (95% CI: 0.615-0.776)
External Validation AUC: 0.565 (95% CI: 0.503-0.627)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with pathology data: 434
After filling missing values: 0 missing values
After filtering - Train: 1473, Test: 259, ExtVal: 240
Train samples: 1473
Test samples: 

# Trimodal Clinical + Radiomics + Digital Pathology

trimodal clinical + pyrad + dp cross validation

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+radiology+pathology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        modality_mask_cv = modality_mask_trimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_pyrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+radiology+pathology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 336
Patients with pathology data: 370
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<


Processing fold 1: 921 train samples, 362 validation samples
 Applying L1 filter on modality position 1
Selected 8 outlier-stable features
 20%|██        | 1/5 [00:06<00:24,  6.03s/it]
Processing fold 2: 773 train samples, 510 validation samples
 Applying L1 filter on modality position 1
Selected 10 outlier-stable features
 40%|████      | 2/5 [00:09<00:14,  4.69s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 10 outlier-stable features
 60%|██████    | 3/5 [00:15<00:10,  5.17s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 Applying L1 filter on modality position 1
Selected 10 outlier-stable features
 80%|████████  | 4/5 [00:21<00:05,  5.50s/it]
Processing fold 5: 1119 train samples, 164 validation samples
 Applying L1 filter on modality position 1
Selected 10 outlier-stable features
100%|██████████| 5/5 [00:27<00:00,  5.47s/it]

Overall AUC calculated on 1283 samples (pos=324, neg=959)
AUC = 0.

trimodal clinical + pyrad + dp train/test/extval split

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+radiology+pathology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled],
            modality_mask=modality_mask_trimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_pyrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+radiology+pathology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 336
Patients with pathology data: 370
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 11 outlier-stable features

Test AUC: 0.700 (95% CI: 0.620-0.781)
External Validation AUC: 0.556 (95% CI: 0.494-0.617)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 374
Pat

trimodal clinical + fmrad + dp cross val

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+FM radiology+pathology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        modality_mask_cv = modality_mask_trimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_fmrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+FM radiology+pathology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 350
Patients with pathology data: 370
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:

trimodal clinical + fmrad + dp train/test/extval split

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+FM radiology+pathology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled],
            modality_mask=modality_mask_trimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_fmrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+FM radiology+pathology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 350
Patients with pathology data: 370
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 2704 outlier-stable features

Test AUC: 0.708 (95% CI: 0.628-0.787)
External Validation AUC: 0.564 (95% CI: 0.502-0.626)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 39

# Multimodal Clinical + Radiomics + Digital Pathology + Genomics

multimodal clinical + pyrad + dp + genomics cross val

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+radiology+pathology+genomics) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        X_gen_cv = X_gen_filled.loc[cv_index]
        modality_mask_cv = modality_mask_quadmodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv, X_gen_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_pyrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+radiology+pathology+genomics) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 336
Patients with pathology data: 370
Patients with genomics data: 654
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] 

multimodal clinical + pyrad + dp + genomics train/test/ext_val

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+radiology+pathology+genomics, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled, X_gen_filled],
            modality_mask=modality_mask_quadmodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_pyrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+radiology+pathology+genomics, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 336
Patients with pathology data: 370
Patients with genomics data: 654
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 11 outlier-stable features

Test AUC: 0.707 (95% CI: 0.627-0.787)
External Validation AUC: 0.594 (95% CI: 0.518-0.669)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data:

multimodal clinical + radfm + dp + genomics cross val

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+FM radiology+pathology+genomics) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        X_gen_cv = X_gen_filled.loc[cv_index]
        modality_mask_cv = modality_mask_quadmodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv, X_gen_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/cv_rwd_fmrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+FM radiology+pathology+genomics) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 350
Patients with pathology data: 370
Patients with genomics data: 654
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC

 40%|████      | 2/5 [01:37<02:26, 49.00s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 3292 outlier-stable features
 60%|██████    | 3/5 [02:28<01:39, 49.99s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 Applying L1 filter on modality position 1
Selected 3375 outlier-stable features
 80%|████████  | 4/5 [03:17<00:49, 49.53s/it]
Processing fold 5: 1119 train samples, 164 validation samples
 Applying L1 filter on modality position 1
Selected 3253 outlier-stable features
100%|██████████| 5/5 [04:08<00:00, 49.60s/it]

Overall AUC calculated on 1283 samples (pos=324, neg=959)
AUC = 0.655 (95% CI: 0.620-0.691)

--- Processing outcome: OS_6 for cohort23 ---
Patients with OS_6 label: 1972
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 390
Patients with pathology data: 434
Patients with genomics data: 819
Predefined folds after alignment: 1473 samples
Folds availab

multimodal clinical + radfm + dp + genomics train/test/ext_val

In [ ]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+FM radiology+pathology+genomics, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled, X_gen_filled],
            modality_mask=modality_mask_quadmodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'].to_csv(
            f'/home/ludovica/vanguri/results/standard_rwd_fmrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+FM radiology+pathology+genomics, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 350
Patients with pathology data: 370
Patients with genomics data: 654
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 2704 outlier-stable features



Test AUC: 0.708 (95% CI: 0.627-0.790)
External Validation AUC: 0.580 (95% CI: 0.516-0.644)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 390
Patients with pathology data: 434
Patients with genomics data: 819
After filling missing values: 0 missing values
After filtering - Train: 1473, Test: 259, ExtVal: 240
Train samples: 1473
Test samples: 259
External validation samples: 240
Applying L1 filter on modality position 1
Selected 2591 outlier-stable features

Test AUC: 0.681 (95% CI: 0.611-0.751)
External Validation AUC: 0.634 (95% CI: 0.552-0.717)


--- Processing outcome: DCR for cohort23 ---
Processing outcome: DCR, initial samples: 2075
After dropping missing labels: 2068 samples
Initial missing values in features: 0
Total patients: 2068
Patients with clinical data: 2068